# Reconia — GNN trên DocILE
Notebook riêng cho bản sửa stable hash. **Lần đầu: smoke test** (16 train / 4 val), không phải benchmark.

Để train đầy đủ: đổi `RUN_MODE = "train"` ở ô cấu hình, chọn **Save Version → Save & Run All**. Giữ GPU và Internet bật. Chỉ dùng một GPU. Dữ liệu đã attach: 4.680 train / 500 val / 500 test nội bộ.

Kết quả được lưu trong `/kaggle/working/reconia-runs` và ZIP kết quả. Test nội bộ không phải test DocILE chính thức. Không dùng checkpoint GNN cũ.


In [ ]:
RUN_MODE = "smoke"  # "smoke" or "train"
RUN_ID = "gnn-docile-seed42-v1"
EPOCHS = 20
SEED = 42
RESUME_FROM = ""  # Attached prior output's checkpoints directory; requires best + latest.


In [ ]:
import os, sys, json, hashlib, shutil, subprocess, zipfile, re
from pathlib import Path
assert RUN_MODE in {"smoke", "train"}
assert re.fullmatch(r"[a-zA-Z0-9_-]+", RUN_ID), "Use a simple run ID"
assert EPOCHS > 0
assert not RESUME_FROM or RUN_MODE == "train", "Resume is only supported for full training"
# Keep Kaggle's existing CUDA PyTorch, NumPy and pandas installations.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "python-dotenv==1.0.1", "transformers==4.57.6"], check=True)
INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
CODE = Path("/kaggle/temp/reconia-code")
DATA = Path("/kaggle/temp/reconia-docile")
def extract_bundle(filename, destination):
    matches = list(INPUT.rglob(filename))
    assert len(matches) == 1, f"Attach exactly one Reconia bundle: expected {filename}, found {matches}"
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(matches[0]) as z:
        for name in z.namelist():
            assert (destination / name).resolve().is_relative_to(destination.resolve())
        z.extractall(destination)
extract_bundle("reconia-code.bin", CODE)
extract_bundle("docile-export.bin", DATA)
provenance = json.loads((CODE / "source-provenance.json").read_text())
for name, digest in provenance["files_sha256"].items():
    assert hashlib.sha256((CODE / name).read_bytes()).hexdigest() == digest, name
os.chdir(CODE)
sys.path.insert(0, str(CODE))
os.environ["DOCILE_EXPORT_DIR"] = str(DATA)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONHASHSEED"] = str(SEED)
import torch
assert torch.cuda.is_available(), "Enable a GPU accelerator in Notebook Settings"
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch:", torch.__version__)
from app.data.portable import ExportAdapter
adapter = ExportAdapter(DATA, "docile")
counts = {s: len(adapter.document_ids(s)) for s in ("train", "val", "test")}
assert counts == {"train": 4680, "val": 500, "test": 500}, counts
print("Verified dataset:", counts)
del adapter


In [ ]:
from datetime import datetime, timezone
OUT = WORK / "reconia-runs" / (RUN_ID + ("-smoke" if RUN_MODE == "smoke" else ""))
CHECKPOINTS = OUT / "checkpoints"
REPORTS = OUT / "reports"
CACHE = Path("/kaggle/temp/reconia-graphs") / OUT.name
if any(CHECKPOINTS.glob("*.pt")):
    raise RuntimeError("This RUN_ID already has checkpoints. Choose a new RUN_ID; use RESUME_FROM to continue.")
CHECKPOINTS.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)
resume_args = []
if RESUME_FROM:
    previous = Path(RESUME_FROM)
    for filename in ("gnn_docile_latest.pt", "gnn_docile_best.pt"):
        assert (previous / filename).is_file(), f"Missing resume artifact: {filename}"
        shutil.copy2(previous / filename, CHECKPOINTS / filename)
    from app.models_dl.gnn import validate_gnn_checkpoint
    saved = torch.load(CHECKPOINTS / "gnn_docile_latest.pt", map_location="cpu", weights_only=False)
    validate_gnn_checkpoint(saved)
    assert saved["epoch"] < EPOCHS, "Already finished; increase EPOCHS only for a new planned schedule"
    assert saved["config"].hidden_dim == 128 and saved["config"].num_layers == 2
    del saved
    resume_args = ["--resume", str(CHECKPOINTS / "gnn_docile_latest.pt")]
command = [sys.executable, "-u", "-m", "app.training.train_gnn",
           "--dataset", "docile", "--device", "cuda", "--seed", str(SEED),
           "--epochs", str(EPOCHS if RUN_MODE == "train" else 1),
           "--lr", "0.001", "--hidden-dim", "128", "--num-layers", "2",
           "--batch-accum", "16", "--patience", "5", "--weighted-loss", "true",
           "--cache-dir", str(CACHE), "--save-dir", str(CHECKPOINTS), "--report-dir", str(REPORTS)]
if RUN_MODE == "smoke":
    command += ["--limit-train", "16", "--limit-val", "4"]
command += resume_args
configuration = {"mode": RUN_MODE, "run_id": RUN_ID, "epochs": EPOCHS, "seed": SEED,
                 "resume_from": RESUME_FROM, "counts": counts, "command": command,
                 "gpu": torch.cuda.get_device_name(0), "started_utc": datetime.now(timezone.utc).isoformat(),
                 "source": provenance, "export_manifest_sha256": hashlib.sha256((DATA / "manifest.json").read_bytes()).hexdigest()}
(OUT / "run-config.json").write_text(json.dumps(configuration, indent=2))
(OUT / "requirements-freeze.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
print("Output:", OUT)
print("Launching:", " ".join(command))
# Train in a child process so stopping the notebook does not hide Python errors.
subprocess.run(command, check=True, cwd=CODE)


In [ ]:
if RUN_MODE == "smoke":
    # Reload weights in a fresh process; use validation, never test for debugging.
    subprocess.run([sys.executable, "-u", "-m", "app.evaluation.predict",
                    "--model", "gnn", "--dataset", "docile", "--split", "val",
                    "--checkpoint", str(CHECKPOINTS / "gnn_docile_best.pt"),
                    "--limit", "4", "--device", "cuda", "--report-dir", str(REPORTS)], check=True, cwd=CODE)
    print("SMOKE PASSED. For the full experiment set RUN_MODE = 'train' and Save & Run All.")
else:
    evidence = REPORTS / "predictions/predictions_gnn_docile_test.metrics.json"
    metrics = json.loads(evidence.read_text())
    assert metrics["complete"] and metrics["num_documents"] == metrics["expected_num_documents"] == 500
    print("Full local test verified:", {k: metrics[k] for k in ("precision", "recall", "f1", "protocol")})
    print("Field-value F1:", metrics["value_metrics"]["f1"])
archive = shutil.make_archive(str(WORK / OUT.name), "zip", root_dir=OUT.parent, base_dir=OUT.name)
from IPython.display import FileLink, display
print("Download and retain both best/latest checkpoints, reports and run configuration.")
os.chdir(WORK)
display(FileLink(str(Path(archive).relative_to(WORK))))


## Sau khi chạy
- Smoke thành công chỉ xác nhận đường chạy. Chuyển `RUN_MODE` sang `train` để thực nghiệm đầy đủ.
- Tải ZIP trong Outputs sau khi phiên Save & Run All hoàn tất. Graph cache không nằm trong ZIP.
- Resume: attach output trước, chỉ rõ `RESUME_FROM` tới thư mục `checkpoints`, đổi RUN_ID, giữ nguyên cấu hình và tổng EPOCHS. Cần cả best và latest; không khôi phục được file chưa được Kaggle lưu.
- Notebook hiện hỗ trợ GNN/DocILE. SROIE/CORD và các mô hình khác cần dữ liệu/cấu hình riêng; bundle OCR này không có ảnh cho VLM.
